In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
# read the data from Bronze

order_items_df = (
    spark.read.table("retail_lakehouse.bronze.order_items")
)

display(order_items_df)

In [0]:
# standardize the column names
from pyspark.sql import functions as F
orders_clean_df = (
    order_items_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )
    .withColumn(
        "line_number",
        F.col("line_number").cast("int")
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("double")
    )

    .withColumn(
        "line_amount",
        F.col("line_amount").cast("double")
    )
)

display(orders_clean_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

order_items_window = Window.partitionBy("order_id", "line_number")

order_items_validation_df = (
    orders_clean_df
    .withColumn(
        "dq_order_id_not_null",
        F.col('order_id').isNotNull()
    )

    .withColumn(
        "dq_product_id_not_null",
        F.col("product_id").isNotNull()
    )

    .withColumn(
        "dq_line_number_not_null",
        F.col("line_number").isNotNull()
    )

    .withColumn(
        "dq_order_line_count",
        F.count("*").over(order_items_window)
    )

    .withColumn(
        "dq_order_line_unique",
        F.col("dq_order_line_count") == 1
    )

    .withColumn(
        "dq_quantity",
        F.col("quantity") > 0
    )

    .withColumn(
        "dq_unit_price",
        F.col("unit_price") > 0
    )
)

In [0]:
# line_amount = quantity * unit_price

order_items_validation_df = (
    order_items_validation_df
    .withColumn(
        "dq_line_amount",
        F.abs(
            F.col("line_amount") - 
            (F.col("quantity") * F.col("unit_price"))
        ) < 0.01
    )
)

display(order_items_validation_df)

In [0]:
# referential integratity check

orders_lookup_df = (
    spark.read.table("retail_lakehouse.sliver.orders")
    .select(
        F.col("order_id").alias("ref_order_id")
    )
    .dropDuplicates()
)

order_id_check_df = (
    order_items_validation_df
    .join(
        orders_lookup_df,
        order_items_validation_df["order_id"] == orders_lookup_df["ref_order_id"], "left"
    )
    .withColumn(
        "dq_order_id_exists",
        F.col("ref_order_id").isNotNull()
    )
)

In [0]:
# product_id lookup
product_lookup_df = (
    spark.read.table("retail_lakehouse.sliver.products")
    .select(
        F.col("product_id").alias("ref_product_id"),  
    )
    .dropDuplicates()
)

product_check_df = (
    order_id_check_df
    .join(
        product_lookup_df,
        order_id_check_df["product_id"] ==
        product_lookup_df["ref_product_id"],
        "left"
    )
    .withColumn(
        "dq_product_id_exists",
        F.col("ref_product_id").isNotNull()
    )
)

In [0]:
order_items_final_df = (
    product_check_df
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_order_id_not_null") &
            F.col("dq_product_id_not_null") &
            F.col("dq_line_number_not_null") &
            F.col("dq_order_line_unique") &
            F.col("dq_quantity") &
            F.col("dq_unit_price") &
            F.col("dq_line_amount") &
            F.col("dq_order_id_exists") &
            F.col("dq_product_id_exists"),
            "PASS"
        ).otherwise("FAIL")
    )

)

In [0]:
display(
    order_items_final_df
    .groupBy("dq_status")
    .count()
)

In [0]:
validation_orders_df =(
    order_items_final_df
    .filter(F.col("dq_status") == "PASS")
)

quarantine_orders_df = (
    order_items_final_df
    .filter(F.col("dq_status") == "FAIL")
)
    
display(validation_orders_df)
display(quarantine_orders_df)


In [0]:
# create a delta table in sliver layer
silver_order_items_df = (
    validation_orders_df
    .select(
        "order_id",
        "line_number",
        "product_id",
        "quantity",
        "unit_price",
        "line_amount"
    )
)


In [0]:
quarantine_order_items_df = (
    order_items_final_df
    .filter(F.col("dq_status") == "FAIL")
)

In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

(
    silver_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("order_items")
)

(
    quarantine_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("order_items_quarantine")
)
